# Gradient Boosting Model
**Course:** M1 Hands-on Machine Learning (2026–27)  
**Author:** Vida Kayla BENSING
**Model Focus:** HistGradientBoostingRegressor

# 1. Intro

### Problem Framing & Objective
We predict single-family residential property sale prices in Cook County, Illinois. This is formalized as a supervised tabular regression problem where the target variable is the continuous transaction price in US dollars.

### State-of-the-Art (SOTA) Benchmark
Our reference benchmark is the Cook County Assessor’s Office production model (`ccao-data/model-res-avm`). While the county historically used linear regression per township, they transitioned to gradient boosted trees (LightGBM) to capture non-linear relationships, spatial interactions, and diminishing marginal value for structural features.

### Why Gradient Boosting?
As covered in lecture, Gradient Boosting trains decision trees sequentially:
$$F_m(x) = F_{m-1}(x) + \eta h_m(x)$$
where each base tree $h_m$ fits the pseudo-residuals of the previous stage using gradient descent with learning rate $\eta$. Unlike Random Forests (which train independent parallel trees via bagging to reduce variance), Gradient Boosting focuses primarily on reducing bias while boosting predictive capacity on complex tabular data.

### Evaluation Scorecard
Following lecture definitions, all models are scored on original dollar scales using:
- **MAE (Mean Absolute Error):** $\frac{1}{n}\sum \vert{}y_i - \hat{y}_i\vert{}$ (median-centered average dollar error)
- **RMSE (Root Mean Squared Error):** $\sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}$ (penalizes large outlier mistakes)
- **$R^2$ (Coefficient of Determination):** $1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ (variance explained relative to the naive mean baseline)

# 2. Preprocessing

### Dataset & Chronological Splitting Protocol
To simulate true out-of-time forecasting and prevent look-ahead bias, the dataset is partitioned chronologically across three non-overlapping periods:
- **Training Set (2017–2023):** 276,685 records (used exclusively to fit preprocessing and model parameters)
- **Validation Set (2024):** 30,545 records (reserved strictly for hyperparameter selection)
- **Test Set (2025):** 31,535 records (untouched holdout for final generalization evaluation)

### Feature Pipeline
The dataset comprises 81 features (25 property traits, 56 contextual/spatial indicators).
To prevent data leakage, imputation and tree encodings (`make_tree_encoder()`) are learned strictly on the 2017–2023 training split:
- Numerical features are median-imputed.
- Categorical features use an explicit `'Missing'` indicator category and integer/ordinal encoding suitable for histogram-based gradient trees.

In [13]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

from shared_preprocessing import load_splits, make_tree_encoder

# the splits
X_train, y_train, X_val, y_val, X_test, y_test = load_splits("../data/stage2_outputs")

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}   | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

X_train shape: (276685, 81) | y_train shape: (276685,)
X_val shape:   (30545, 81)   | y_val shape:   (30545,)
X_test shape:  (31535, 81)  | y_test shape:  (31535,)


In [14]:
def evaluate_predictions(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"--- {name} ---")
    print(f"MAE:  ${mae:,.2f}")
    print(f"RMSE: ${rmse:,.2f}")
    print(f"R2:   {r2:.4f}\n")
    
    return {"model": name, "mae": mae, "rmse": rmse, "r2": r2}

# 3. Training

We evaluate models progressively:
1. **Naive Mean Baseline:** Predicts the historical training mean $\bar{y}_{\text{train}}$, providing the minimum performance floor ($R^2 \approx 0$).
2. **Default HistGradientBoosting:** Scikit-learn default configuration as an initial reference.
3. **Hyperparameter Tuning:** Grid exploration over learning rate $\eta \in [0.05, 0.10]$, maximum iterations (trees) $\in [200, 300]$, and maximum depth $\in [8, 12]$. Selection is guided strictly by validation RMSE.

In [15]:
# simple mean baseline
train_mean = y_train.mean()
print(f"Train mean price: ${train_mean:,.2f}\n")

val_baseline_preds = np.full_like(y_val, fill_value=train_mean)
baseline_val_res = evaluate_predictions("Mean Baseline (Val 2024)", y_val, val_baseline_preds)

Train mean price: $350,713.99

--- Mean Baseline (Val 2024) ---
MAE:  $227,823.97
RMSE: $444,016.55
R2:   -0.0463



In [16]:
# initial HistGradientBoosting model

gb_pipe = make_pipeline(
    make_tree_encoder(),
    HistGradientBoostingRegressor(random_state=42)
)

gb_pipe.fit(X_train, y_train)

val_preds_default = gb_pipe.predict(X_val)
gb_default_res = evaluate_predictions("HistGB - Default (Val 2024)", y_val, val_preds_default)

--- HistGB - Default (Val 2024) ---
MAE:  $114,852.25
RMSE: $196,645.88
R2:   0.7948



In [17]:
# hyperparameter tuning (validation set)

tuning_configs = [
    {"learning_rate": 0.05, "max_iter": 200, "max_depth": 8},
    {"learning_rate": 0.1,  "max_iter": 250, "max_depth": 10},
    {"learning_rate": 0.08, "max_iter": 300, "max_depth": 12},
]

best_score = -float("inf")
best_model = None
best_params = None

for cfg in tuning_configs:
    pipe = make_pipeline(
        make_tree_encoder(),
        HistGradientBoostingRegressor(
            learning_rate=cfg["learning_rate"],
            max_iter=cfg["max_iter"],
            max_depth=cfg["max_depth"],
            random_state=42
        )
    )
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)
    r2 = r2_score(y_val, preds)
    mae = mean_absolute_error(y_val, preds)
    print(f"params: {cfg} -> Val R2: {r2:.4f}, MAE: ${mae:,.2f}")
    
    if r2 > best_score:
        best_score = r2
        best_model = pipe
        best_params = cfg

print(f"\nBest config: {best_params} (Val R2 = {best_score:.4f})")

params: {'learning_rate': 0.05, 'max_iter': 200, 'max_depth': 8} -> Val R2: 0.7931, MAE: $114,757.63
params: {'learning_rate': 0.1, 'max_iter': 250, 'max_depth': 10} -> Val R2: 0.8162, MAE: $107,526.22
params: {'learning_rate': 0.08, 'max_iter': 300, 'max_depth': 12} -> Val R2: 0.8142, MAE: $108,344.73

Best config: {'learning_rate': 0.1, 'max_iter': 250, 'max_depth': 10} (Val R2 = 0.8162)


In [18]:
tuning_df = pd.DataFrame(tuning_configs) 
tuning_df.to_csv("../results/gb_validation_tuning.csv", index=False)
print("Saved to ../results/gb_validation_tuning.csv")

Saved to ../results/gb_validation_tuning.csv


# 4. Evaluation

We evaluate the selected model on the unseen 2025 holdout test set (31,535 sales):
1. **Overall Performance:** Final test scores (MAE, RMSE, $R^2$) compared to the baseline.
2. **Domain Boundary Check:** Verify that predictions remain non-negative.
3. **Price Tier Error Analysis:** Inspect signed error ($y_{\text{pred}} - y_{\text{true}}$) and MAE across four tiers (Up to \$250k, \$250k–\$500k, \$500k–\$1M, Above \$1M).

In [19]:
# final evaluation on 2025 test set
test_preds = best_model.predict(X_test)
gb_test_res = evaluate_predictions("HistGB (Test 2025)", y_test, test_preds)

test_baseline_preds = np.full_like(y_test, fill_value=train_mean)
baseline_test_res = evaluate_predictions("Mean Baseline (Test 2025)", y_test, test_baseline_preds)

--- HistGB (Test 2025) ---
MAE:  $125,945.35
RMSE: $260,565.96
R2:   0.7655

--- Mean Baseline (Test 2025) ---
MAE:  $249,127.97
RMSE: $551,842.59
R2:   -0.0518



In [20]:
import json

final_metrics = pd.DataFrame([
    {"split": "Validation 2024", "model": "HistGB", "MAE": 107526.22, "RMSE": 196645.88, "R2": 0.8162},
    {"split": "Test 2025", "model": "HistGB", "MAE": gb_test_res["mae"], "RMSE": gb_test_res["rmse"], "R2": gb_test_res["r2"]}
])
final_metrics.to_csv("../results/gb_metrics.csv", index=False)
print("Saved to ../results/gb_metrics.csv")

with open("../results/gb_config.json", "w") as f:
    json.dump(best_params, f, indent=4)
print("Saved to ../results/gb_config.json")

Saved to ../results/gb_metrics.csv
Saved to ../results/gb_config.json


### Notes on Results

- The mean baseline gets a negative R² (-0.0518 on 2025 test) because market prices shifted upward compared to the 2017–2023 training mean.
- The tuned HistGradientBoosting model (`lr=0.1`, `max_iter=250`, `max_depth=10`) performed best on the validation set ($R^2 = 0.8162$). Going deeper to `max_depth=12` slightly reduced the score ($0.8142$), likely beginning to overfit.
- On the unseen 2025 test set, the final model achieved $R^2 = 0.7655$ and cut the baseline MAE roughly in half (from ~$249k down to ~$126k).
- The drop from validation (0.816) to test (0.765) is reasonable given the chronological split, reflecting price drift and inflation between 2024 and 2025.

In [22]:
# error analysis by price band

import pandas as pd
import numpy as np

# 1. create evaluation dataframe using 2025 test actuals and predictions
df_err = pd.DataFrame({
    "actual": y_test,
    "pred": test_preds
})

df_err["abs_error"] = (df_err["actual"] - df_err["pred"]).abs()
df_err["raw_error"] = df_err["pred"] - df_err["actual"]  # or actual - pred depending on convention

# 2. define the exact price bands used by Yuming
bins = [0, 250000, 500000, 1000000, np.inf]
labels = ["Up to 250k", "250k–500k", "500k–1M", "Above 1M"]
df_err["price_band"] = pd.cut(df_err["actual"], bins=bins, labels=labels)

# 3. aggregate MAE and Mean Error per band
gb_band_analysis = df_err.groupby("price_band", observed=False).agg(
    test_mae=("abs_error", "mean"),
    mean_error=("raw_error", "mean"),
    count=("actual", "count")
).reset_index()

# 4. check for negative price predictions
num_negative = (df_err["pred"] < 0).sum()
print(f"Negative price predictions in HistGB: {num_negative}")
display(gb_band_analysis)

Negative price predictions in HistGB: 0


,price_band,test_mae,mean_error,count
0,Up to 250k,54610.877104,-5218.655431,8694
1,250k–500k,92595.735549,-82769.966848,14348
2,500k–1M,174848.236459,-162282.519855,6070
3,Above 1M,456874.683155,-421124.564517,2423


In [23]:
import os

os.makedirs("../results", exist_ok=True)
gb_band_analysis.to_csv("../results/gb_errors_by_price_band.csv", index=False)
print("Saved to ../results/gb_errors_by_price_band.csv")

Saved to ../results/gb_errors_by_price_band.csv


### Notes on Price Tier Errors

- No negative predictions: HistGB produced zero negative prices (ridge produced 214), respecting real-world non-negative price bounds.
- Lower errors across all tiers: Boosting outperforms ridge across every bracket, cutting test mae on homes under $250k by ~$15k and on luxury properties (>$1m) by ~$148k.
- High-end underprediction: Both models underpredict expensive homes (histgb averages -$421k on >$1m homes) because sparse luxury sales create a heavy right-skewed tail that tree ensembles pull toward denser price ranges.

# 5. Conclusion

### What We Did
We implemented, tuned, and evaluated `HistGradientBoostingRegressor` on over 338,000 Cook County residential sales across an out-of-time chronological split (2017–2023 train, 2024 validation, 2025 test).

### What Worked
- **Substantial Error Reduction:** The tuned model (`lr=0.10`, `max_iter=250`, `max_depth=10`) achieved a 2025 Test $R^2$ of **0.7655** and MAE of **$125,945**, cutting naive test error roughly in half ($49.4\%$ reduction) and substantially outperforming the regularized linear model (Ridge $R^2 = 0.5817$).
- **Realistic Physical Predictions:** The model produced **0 negative predictions** across all 31,535 test properties, whereas Ridge produced 214 negative values.
- **Handling Geographic Coordinates:** Decision tree threshold splits effectively partition spatial coordinates (`latitude`, `longitude`) into local neighborhood price zones without manual spatial feature engineering.

### What Didn't (Limitations)
- **High-End Luxury Underprediction:** Properties selling above \$1M had an average signed error of **-$421,125$** (underprediction). Real estate data is heavily right-skewed; because high-value sales are sparse, tree leaf nodes pull extreme luxury sales toward denser central pricing ranges.
- **Chronological Generalization Drop:** $R^2$ dropped from 0.8162 on the 2024 validation set to 0.7655 on the 2025 test set, reflecting out-of-time macroeconomic inflation and price drift.

### What We Would Do With More Time
1. **Target Transformations:** Train the boosting pipeline on log-transformed prices ($\log(\text{price})$) to penalize relative percentage errors and dampen luxury right-tail skew.
2. **Spatial Splines / Neighborhood Target Encodings:** Incorporate target-encoded geographic boundaries and spatial smoothing splines.
3. **Monotonic Constraints:** Enforce monotonic constraints on key features (e.g., ensuring square footage and bathroom counts never decrease estimated property value).